In [ ]:
import duckdb
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
from sklearn.feature_selection import mutual_info_regression


# run utils.db_setup to set up duckdb database
conn = duckdb.connect('f1.db')
%load_ext sql
%sql conn --alias duckdb
%config SqlMagic.displaycon=False
%config SqlMagic.autopandas=True

# graphs too large for interactivity
pio.renderers.default = 'svg'

## Schema

![](f1_schema.png)

## Tasks

### `rel-f1-driver-position`

Task is to predict the average finishing position of each driver all races in the next 2 months.

Regression

Metric: MAE

In [ ]:
%%sql train_df <<
from driver_position_train

In [ ]:
train_df.head()

In [ ]:
print('TRAIN\n')

print(f'{len(train_df):,} samples over {train_df.driverId.nunique():,} drivers.')
print()
print('Temporal dist of samples:')
months = train_df.date.dt.strftime('%Y-%m')
print(months.value_counts(normalize=True).sort_index())
print()
print('Temporal dist of drivers (first appearance):')
print(
    # date is sorted in desc order ==> keep='last'
    train_df
    .drop_duplicates(subset='driverId', keep='last')
    .groupby(months).driverId.nunique() / train_df.driverId.nunique()
)
print()
print('Label Distribution:')
mean = train_df.position.mean()
std = train_df.position.std()
print(f'Mean (std): {mean:,.1f} ± {std:,.1f}')
print(f'Prop zeros: {(train_df.position == 0).mean():.1%}')
print(f'Quantiles: \n{train_df.position.quantile(np.arange(0, 1.1, 0.1))}')

In [ ]:
%%sql val_df <<
from driver_position_val

In [ ]:
print('VAL\n')

print(f'{len(val_df):,} samples over {val_df.driverId.nunique():,} drivers.')
print()
print('Temporal dist of samples:')
months = val_df.date.dt.strftime('%Y-%m')
print(months.value_counts(normalize=True).sort_index())
print()
print('Temporal dist of drivers (first appearance):')
print(
    # date is sorted in desc order ==> keep='last'
    val_df
    .drop_duplicates(subset='driverId', keep='last')
    .groupby(months).driverId.nunique() / val_df.driverId.nunique()
)
print()
print(f'Drivers in val seen in train: {len(set(val_df.driverId) & set(train_df.driverId)):,}')
print()
print('Label Distribution:')
mean = val_df.position.mean()
std = val_df.position.std()
print(f'Mean (std): {mean:,.1f} ± {std:,.1f}')
print(f'Prop zeros: {(val_df.position == 0).mean():.1%}')
print(f'Quantiles: \n{val_df.position.quantile(np.arange(0, 1.1, 0.1))}')

### `rel-f1-driver-dnf`

Task is to predict whether a driver will DNF (not finish) a race in the next 1 month.

Binary classification

Metric: ROCAUC

In [ ]:
%%sql train_df <<
from driver_dnf_train

In [ ]:
print('TRAIN\n')

print(f'{len(train_df):,} samples over {train_df.driverId.nunique():,} drivers.')
print()
print('Label Distribution:')
print(train_df.did_not_finish.value_counts(normalize=True))
print()
print('Temporal dist of samples:')
months = train_df.date.dt.strftime('%Y-%m')
print(months.value_counts(normalize=True).sort_index())
print()
print('Temporal dist of drivers (first appearance):')
print(
    # date is sorted in desc order ==> keep='last'
    train_df
    .drop_duplicates(subset='driverId', keep='last')
    .groupby(months).driverId.nunique() / train_df.driverId.nunique()
)

In [ ]:
%%sql val_df <<
from driver_dnf_val

In [ ]:
print('VAL\n')

print(f'{len(val_df):,} samples over {val_df.driverId.nunique():,} drivers.')
print()
print('Label Distribution:')
print(val_df.did_not_finish.value_counts(normalize=True))
print()
print('Temporal dist of samples:')
months = val_df.date.dt.strftime('%Y-%m')
print(months.value_counts(normalize=True).sort_index())
print()
print(f'Drivers in val seen in train: {len(set(val_df.driverId) & set(train_df.driverId)):,}')
print()
print('Temporal dist of drivers (first appearance):')
print(
    # date is sorted in desc order ==> keep='last'
    val_df
    .drop_duplicates(subset='driverId', keep='last')
    .groupby(months).driverId.nunique() / val_df.driverId.nunique()
)

### `rel-f1-driver-top3`

Task is to predict if each driver will qualify in the top-3 for a race within the next 1 month.

Binary classification

Metric: ROCAUC

In [ ]:
%%sql train_df <<
from driver_top3_train

In [ ]:
print('TRAIN\n')

print(f'{len(train_df):,} samples over {train_df.driverId.nunique():,} drivers.')
print()
print('Label Distribution:')
print(train_df.qualifying.value_counts(normalize=True))
print()
print('Temporal dist of samples:')
months = train_df.date.dt.strftime('%Y-%m')
print(months.value_counts(normalize=True).sort_index())
print()
print('Temporal dist of drivers (first appearance):')
print(
    # date is sorted in desc order ==> keep='last'
    train_df
    .drop_duplicates(subset='driverId', keep='last')
    .groupby(months).driverId.nunique() / train_df.driverId.nunique()
)

In [ ]:
%%sql val_df <<
from driver_top3_val

In [ ]:
print('VAL\n')

print(f'{len(val_df):,} samples over {val_df.driverId.nunique():,} drivers.')
print()
print('Label Distribution:')
print(val_df.qualifying.value_counts(normalize=True))
print()
print('Temporal dist of samples:')
months = val_df.date.dt.strftime('%Y-%m')
print(months.value_counts(normalize=True).sort_index())
print()
print(f'Drivers in val seen in train: {len(set(val_df.driverId) & set(train_df.driverId)):,}')
print()
print('Temporal dist of drivers (first appearance):')
print(
    # date is sorted in desc order ==> keep='last'
    val_df
    .drop_duplicates(subset='driverId', keep='last')
    .groupby(months).driverId.nunique() / val_df.driverId.nunique()
)

## Circuits

In [ ]:
%%sql circuits <<
from circuits

In [ ]:
circuits.head()

In [ ]:
circuits.location.value_counts()

## Races

In [ ]:
%%sql
from races

In [ ]:
%%sql label_periods <<
-- how many races in 1 month label periods
with timestamps as (
    select distinct date
    from position_train
)
select
    timestamps.date,
    count(*) as num_races,
    min(round) as starting_round,
    max(round) as ending_round,
    list(name order by races.date asc) as race_list
from timestamps
left join races
    on timestamps.date <= races.date
    and timestamps.date + interval '1 month' > races.date
group by timestamps.date
order by timestamps.date desc

In [ ]:
label_periods.sample(10)

In [ ]:
label_periods.num_races.describe()

In [ ]:
label_periods.starting_round.describe()

In [ ]:
label_periods.starting_round.quantile(np.arange(0, 1.1, 0.1))

In [ ]:
all_races = sum(label_periods.race_list.apply(list).tolist(), [])

In [ ]:
px.histogram(x=all_races)

## Constructors

In [ ]:
%%sql
from constructors

## Drivers

In [ ]:
%%sql drivers <<
from drivers

In [ ]:
drivers.head()

In [ ]:
drivers.nationality.value_counts()

## Constructor Standings

In [ ]:
%%sql
from constructor_standings

In [ ]:
%%sql
from constructor_standings
where constructorId = 0
order by raceId desc

## Constructor Results

In [ ]:
%%sql
from constructor_results

## Standings

In [ ]:
%%sql
from standings

## Results

In [ ]:
%%sql
from results

In [ ]:
%%sql
-- duplicates!
select
    raceId,
    driverId,
    count(*) as count
from results
group by raceId, driverId
having count > 1

In [ ]:
%%sql dnfs_over_time <<
select
    strftime('%Y-%m', date) as month,
    avg((statusId != 1)::int) as dnfs
from results
group by month


In [ ]:
px.line(dnfs_over_time.sort_values(by='month'), x='month', y='dnfs')

In [ ]:
%%sql dnfs_over_time <<
select
    raceId,
    any_value(date_part('year', date)) as year,
    avg((statusId != 1)::int) as dnfs
from results
group by raceId

In [ ]:
px.box(dnfs_over_time, x='year', y='dnfs')

In [ ]:
%%sql dnfs_over_time <<
select
    raceId,
    any_value(date_part('year', date)) as year,
    avg((statusId = 1)::int) as dnfs
from results
group by raceId

In [ ]:
%%sql accidents <<
select
    raceId,
    any_value(date_part('year', date)) as year,
    sum((statusId in (3, 4))::int) as accident_or_collision
from results
group by raceId

In [ ]:
px.box(accidents, x='year', y='accident_or_collision')

In [ ]:
%%sql
select
    statusId,
    count(*) as count,
    count(*) / sum(count(*)) over () as prop
from results
group by statusId
order by prop desc

## Qualifying

In [ ]:
%%sql
from qualifying

In [ ]:
conn.close()